# Figure 1: EndToEndPhysicsUnrollSPECTNet Architecture & Pipeline
## 端到端物理展开 SPECT 重建网络架构图

本 Notebook 生成论文 **Figure 1**：
端到端联合物理展开重建网络（`EndToEndPhysicsUnrollSPECTNet`）的系统流程与物理算子架构。

### 核心物理设计要点：
1. **双能窗投影输入**：光峰投影 $Y_{primary}$ 与散射窗投影 $Y_{scatter}$ 结合采集几何条件 $\mathbf{c} = [dr, radii]$；
2. **渐进衰减图估计分支**：初始粗略 $\mu$ 估计与级联 ProxNet 迭代细化；
3. **关键梯度解耦机制 (`mu.detach()`)**：切断泊松投影损失对骨骼衰减系数的负向梯度拉扯（彻底解决历史退化问题）；
4. **自适应残差物理展开**：前向投影 $P_{\mu}$ 与反向投影 $P_{\mu}^T$ 算子结合残差 3D U-Net 与可学习阻尼标量 $lpha_k$；
5. **多任务物理联合损失**：活度相对 L1 损失 + 组织均衡衰减损失 + 泊松投影一致性损失。


In [ ]:
import os
import sys
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# 设置论文出版级字体与样式
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 10
plt.rcParams['axes.linewidth'] = 1.2
plt.rcParams['pdf.fonttype'] = 42

fig, ax = plt.subplots(figsize=(14, 8), dpi=300)
ax.set_xlim(0, 14)
ax.set_ylim(0, 8)
ax.axis('off')

# 辅助绘制圆角矩形
def add_box(ax, xy, w, h, title, subtitle="", color="#EBF3FB", border="#1E6BB8", lw=1.5):
    rect = patches.FancyBboxPatch(
        xy, w, h,
        boxstyle="round,pad=0.15",
        ec=border, fc=color, lw=lw, zorder=2
    )
    ax.add_patch(rect)
    cx = xy[0] + w / 2
    cy = xy[1] + h / 2
    if subtitle:
        ax.text(cx, cy + 0.18, title, ha='center', va='center', fontsize=11, fontweight='bold', color='#1A202C')
        ax.text(cx, cy - 0.22, subtitle, ha='center', va='center', fontsize=8.5, color='#4A5568')
    else:
        ax.text(cx, cy, title, ha='center', va='center', fontsize=11, fontweight='bold', color='#1A202C')
    return cx, cy

def add_arrow(ax, p1, p2, label="", color="#2B6CB0", lw=1.8):
    ax.annotate(
        "", xy=p2, xytext=p1,
        arrowprops=dict(arrowstyle="-|>", color=color, lw=lw, mutation_scale=14),
        zorder=3
    )
    if label:
        mx = (p1[0] + p2[0]) / 2
        my = (p1[1] + p2[1]) / 2 + 0.16
        ax.text(mx, my, label, ha='center', va='center', fontsize=8.5, fontweight='bold', color=color)

# 1. 输入数据区域
add_box(ax, (0.5, 5.0), 2.2, 1.8, "Dual-Window SPECT", "Primary: Y_p (N×128²)\nScatter: Y_s (N×128²)", "#EDF2F7", "#4A5568")
add_box(ax, (0.5, 2.0), 2.2, 1.4, "Acquisition Cond", "c = [dr_xyz, mean_r/100]\n(4-dim vector)", "#EDF2F7", "#4A5568")

# 2. 衰减估计模块 (Mu Estimation Branch)
add_box(ax, (3.5, 5.2), 2.5, 1.6, "DualWindowMuInitNet", "3D Dense U-Net + Softplus\nλ_init_p & λ_init_s", "#FEFCBF", "#D69E2E")
add_box(ax, (3.5, 2.0), 2.5, 1.4, "Decoupled Gradient", "mu_next = mu.detach()\nPrevents Negative Pull!", "#FED7D7", "#E53E3E", lw=2.0)

# 3. 展开阶段 Stage 1..K (Unrolled Stages)
stage_box = patches.FancyBboxPatch((6.8, 1.5), 4.2, 5.8, boxstyle="round,pad=0.2", ec="#805AD5", fc="#FAF5FF", lw=2, linestyle="--", zorder=1)
ax.add_patch(stage_box)
ax.text(8.9, 7.0, "Physics-Unrolled Stage k ∈ {1, 2, 3, 4}", ha='center', va='center', fontsize=12, fontweight='bold', color="#6B46C1")

add_box(ax, (7.1, 5.2), 3.6, 1.4, "Physical Forward-Backward", "OSEM Update Step:\nλ_osem = OSEM_Step(λ^(k-1), P(μ^(k)))", "#E9D8FD", "#805AD5")
add_box(ax, (7.1, 3.4), 3.6, 1.3, "Adaptive Proximal U-Net", "R_λ = ProxResUNet3D(λ_osem, c)\nZero-Init Conv Out", "#D6BCFA", "#6B46C1")
add_box(ax, (7.1, 1.8), 3.6, 1.1, "Damped State Update", "λ^(k) = λ_osem + α_k * R_λ\nα_k = 0.5 * σ(w_k)", "#E9D8FD", "#805AD5")

# 4. 最终输出与监督损失
add_box(ax, (11.6, 5.0), 2.0, 1.8, "Output Activity", "x_final = λ^(4)\nPSNR: 50.3 dB\nSSIM: 0.986", "#C6F6D5", "#38A169")
add_box(ax, (11.6, 2.0), 2.0, 1.8, "Output Mu-Map", "μ_final = μ^(4)\nPSNR: 23.9 dB\nSoft MAE: 0.020", "#BEE3F8", "#3182CE")

# 连接箭头
add_arrow(ax, (2.7, 5.9), (3.5, 5.9), "Dual Proj")
add_arrow(ax, (2.7, 2.7), (3.5, 2.7), "Cond c")
add_arrow(ax, (4.75, 5.2), (4.75, 3.4), "μ_init")
add_arrow(ax, (6.0, 2.7), (7.1, 2.7), "Detached μ")
add_arrow(ax, (6.0, 5.9), (7.1, 5.9), "Y_p, Y_s")
add_arrow(ax, (8.9, 5.2), (8.9, 4.7))
add_arrow(ax, (8.9, 3.4), (8.9, 2.9))
add_arrow(ax, (10.7, 5.9), (11.6, 5.9), "λ^(4)")
add_arrow(ax, (10.7, 2.9), (11.6, 2.9), "μ^(4)")

# 标题
ax.text(7.0, 7.6, "EndToEndPhysicsUnrollSPECTNet: Joint True Physical Unrolled Reconstruction Pipeline",
        ha='center', va='center', fontsize=14, fontweight='bold', color="#1A202C")

# 保存图片
out_png = os.path.join(os.path.dirname(os.path.abspath('__file__')), "figures", "Figure1_architecture_pipeline.png")
out_pdf = os.path.join(os.path.dirname(os.path.abspath('__file__')), "figures", "Figure1_architecture_pipeline.pdf")
os.makedirs(os.path.dirname(out_png), exist_ok=True)
plt.tight_layout()
plt.savefig(out_png, dpi=300, bbox_inches='tight')
plt.savefig(out_pdf, bbox_inches='tight')
print(f"Figure 1 成功生成并保存至: {out_png}")
